# 键值缓存

前面两章，我们探讨了怎样控制模型生成：采样策略和停止条件。接下来，研究一下怎样提升模型推理的性能。

GPT 模型自回归生成有一个性能瓶颈：每生成一个新词元，都要重新计算所有词元的注意力。假设已经生成了 100 个词元，要生成第 101 个词元就要计算 101 个词元的注意力，实践复杂度为 $O(n^2)$。

## 键值缓存

**键值缓存**（KV Cache）的和姓思想：把已生成词元的 key 和 value 缓存起来。生成新词元时只计算 query，再和缓存的 key 和 value 一起做注意力。

### 实现原理

自注意力中，每个词元的 key 和 value 只取决于它自身，与后续词元无关。因此：

* 已生成词元的 key 和 value 可以缓存，不需要重复计算；
* 新词元只需要计算自己的 query，和缓存的 key 和 value 一起做注意力；
* 缓存随着生成过程不断增加，但是每个词元只计算一次。

### 实现要点

* **缓存结构**：每个 Transformer 模块存一份 key 和 value，格式为 `(批, 头数, 已生成长度, 每头嵌入)`；
* **增量计算**：新词元的 key 和 value 和缓存拼接，再计算注意力；
* **位置编码**：新词元的位置编码要加上偏移量（已生成长度）。

In [1]:
import time
from pathlib import Path

import requests

from src.core import *
from src.gpt import *

In [2]:
np.random.seed(42)

## 模型

### GPT 缓存嵌入层

**GPT 缓存嵌入层**需要支持位置偏移：新词元的位置编码从 offset 开始，而不是从 0 开始。

In [3]:
class GPTEmbeddingWithCache(GPTEmbedding):

    def forward(self, x: Tensor, offset=0):
        token = self.embedding(x)
        position = self.positional_embedding(Tensor(range(offset, offset + x.shape[1])))
        return self.dropout(token + position)

### GPT 缓存注意力层

**GPT 缓存注意力层**是键值缓存的核心：

* **无缓存**：正常计算，返回 key 和 value 作为缓存；
* **有缓存**：新词元的 key 和 value 和缓存拼接，再计算注意力。

有缓存时不要因果掩码，因为新词元只与已生成词元计算注意力，天然满足因果关系。

In [4]:
class GPTAttentionWithCache(GPTAttention):

    def forward(self, x: Tensor, use_cache=False, cache=None):
        norm = self.normalize(x)
        head_embedding_size = self.embedding_size // self.heads
        multi_head_shape = (-1, x.shape[1], self.heads, head_embedding_size)
        shape = (-1, x.shape[1], self.embedding_size)

        query = self.query(norm).reshape(multi_head_shape).transpose((0, 2, 1, 3))
        key = self.key(norm).reshape(multi_head_shape).transpose((0, 2, 3, 1))
        value = self.value(norm).reshape(multi_head_shape).transpose((0, 2, 1, 3))

        if use_cache and cache is not None:
            key = Tensor(np.concatenate([cache[0].data, key.data], axis=3))
            value = Tensor(np.concatenate([cache[1].data, value.data], axis=2))
        new_cache = (key, value) if use_cache else None

        scale = Tensor(np.array(1.0 / np.sqrt(head_embedding_size)))
        scores = query @ key * scale
        if cache is None:
            scores = self.mask(scores)
        weights = self.softmax(scores)
        output = self.output((weights @ value).transpose((0, 2, 1, 3)).reshape(shape))
        return x + self.dropout(output), new_cache


### GPT 缓存转换器模块

**GPT 缓存转换器模块**把缓存作为参数，把注意力层的缓存结果传递给前馈网络。

In [5]:
class GPTTransformerWithCache(Composite):

    def __init__(self, embedding_size, heads):
        self.attention = GPTAttentionWithCache(embedding_size, heads)
        self.feed_forward = GPTFeedForward(embedding_size)

        super().__init__([self.attention,
                          self.feed_forward])

    def forward(self, x: Tensor, use_cache=False, cache=None):
        x, new_cache = self.attention(x, use_cache, cache)
        return self.feed_forward(x), new_cache

### GPT 缓存模型

**GPT 缓存模型**管理所有 Transformer 模块的缓存：

* 每个模块一份缓存，用列表存储；
* 前向传播时逐模块传递缓存，收集新的缓存。

In [6]:
class GPTWithCache(Composite):

    def __init__(self, vocab_size, context_size, embedding_size, heads, blocks):
        self.embedding = GPTEmbeddingWithCache(vocab_size, context_size, embedding_size)
        self.transformers = [GPTTransformerWithCache(embedding_size, heads) for _ in range(blocks)]
        self.output = GPTOutput(embedding_size, vocab_size)

        super().__init__([self.embedding] + self.transformers + [self.output])

    def forward(self, x: Tensor, use_cache=False, caches=None, offset=0):
        x = self.embedding(x, offset)
        new_caches = []
        for i, layer in enumerate(self.transformers):
            cache = caches[i] if caches is not None else None
            x, new_cache = layer(x, use_cache, cache)
            new_caches.append(new_cache)
        return self.output(x), new_caches

## GPT 缓存训练器

**GPT 缓存训练器** GPTTrainerWithCache 的 generate 函数：

* **首次生成**：无缓存，计算全部词元；
* **后续生成**：有缓存，只计算新词元；
* **缓存重置**：如果序列长度超过上下文尺寸，重置缓存。

In [7]:
class GPTTrainerWithCache(GPTTrainer):

    def generate(self, dataset, prompt, steps=256, use_cache=True):
        self.model.eval()
        tokens = dataset.encode(prompt)
        caches = None

        with Tensor.no_grad():
            for _ in range(steps):
                if use_cache and caches is not None and len(tokens) <= dataset.context_size:
                    feature = Tensor([tokens[-1:]])
                    offset = len(tokens) - 1
                else:
                    feature = Tensor([tokens[-dataset.context_size:]])
                    offset = 0
                    caches = None

                logits, caches = self.model(feature, use_cache, caches, offset)

                last_logits = logits.data[0, -1]
                exp = np.exp(last_logits - np.max(last_logits))
                probs = exp / np.sum(exp)
                token = np.random.choice(len(probs), p=probs)
                tokens.append(token)

        return dataset.decode(tokens)

## 超参数

### 模型参数

模型参数需要保持和预训练阶段一致。

In [8]:
BATCH_SIZE = 16
CONTEXT_SIZE = 256
EMBEDDING_SIZE = 384
HEADS = 6
BLOCKS = 6

## 建模

### 加载模型

In [9]:
DATA_FILE = "../../../../data/tinyshakespeare.txt"
MODEL_FILE = "../../../../data/tinyshakespeare-gpt.npz"

dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = GPTWithCache(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
loss_fn = GPTLoss()
optimizer = AdamWOptimizer(model.parameters)
trainer = GPTTrainerWithCache(model, loss_fn, optimizer)
trainer.load(MODEL_FILE)

## 无缓存

In [10]:
np.random.seed(42)

start = time.time()
print(trainer.generate(dataset, prompt="ROMEO:", use_cache=False))
print(f"{time.time() - start:.8f} seconds")

ROMEO:
Which the word for neardnath peattle side uponer with.

DIIIIO:
Net you ther gethe herowarrunst,
A Jreath! I gly my crows allet me alt morty lead;
And Unve. Come maugh mod!
Her, I Mys mus.

SheS:
Whind th I Cand ard themage.
Whouring sh
Thown Crews:
Ird m
4.32607913 seconds


## 有缓存

In [11]:
np.random.seed(42)

start = time.time()
print(trainer.generate(dataset, prompt="ROMEO:", use_cache=True))
print(f"{time.time() - start:.8f} seconds")

ROMEO:
YMICC:OIL
YOCCCEIIEIBEHIOCII'IC:YYOE:KIAI TCKEIICYOWTIV:C!EHCOHDIBO:eOC
OLMO:HAOKE:EEMKSIALOIOIII A KEITCHOC:ECVOKOOCTIOTEACIOO
IHCAEWEILHYYCIEE II:DTCCIcCKOCMHKKI:OEC IK
ICKCKHWBFAVPCKOIIC:TTKEGMTSOK:CTI
AK
CIKKCLCEOKOKI:HCCZHTKOIIICMD
KCWYUH
VIYYOENTENT
0.53992009 seconds


本章，我们实现了**键值缓存**：把已生成词元的 key 和 value 缓存起来。新词元只计算和缓存的注意力。时间复杂度从 $O(n^2)$ 降到 $O(n)$，生成速度大幅提升。

## 课后练习

调整最大生成长度 steps，观察**无缓存**和**有缓存**的不同的速度差异。是否生成长度越长，缓存的优势越明显？